## Prepare dictionary model for ET-OptME

In [2]:
import sys
%run  './script/pyomo_solving.py'
import os
import pandas as pd
import cobra
import json
import copy
from cobra import Model, Reaction, Metabolite

## Set path for models and data

In [2]:
# data 
model0_file = '../file/iCW/iCW773_uniprot_modification_del.json' # model with reversible reactions
reaction_kcat_MW_file= '../file/reaction_change_by_enzuse_PDH_n.csv'  # kcat and MW data
# results
path_results = '../results'  # results saving path
dictionarymodel_path= '../results/dictionarymodel_iCW.json'  # dictionary model saving path
# case study
model_file= "../file/lys.json"  # c.glutamicum model, irreversible and reactions with isoezyme splited
reaction_g0_file=os.path.join('../file/iCW773_uniprot_modification_del_reaction_g0.csv') #  standard Gibbs free energy files
metabolites_lnC_file = os.path.join('../file/metabolites_lnC_cg1.txt') # concentration range files

## load and convert the model to dictionary model

In [3]:

model=cobra.io.load_json_model(model_file)
dictionary_model=trans_model2standard_json_etgem(model_file)
[reaction_list,metabolite_list,lb_list,ub_list,coef_matrix]=get_data(model)

Set parameter Username
Academic license - for non-commercial use only - expires 2099-12-31
./lys_irreversible.json


In [4]:
model.reactions.KARA2

Reaction identifier,KARA2
Name,ketol-acid reductoisomerase (2-Acetolactate)
Memory address,0x7493e34172e0
Stoichiometry,"2ahbut_c + h_c + nadph_c --> 23dhmp_c + nadp_c (S)-2-Aceto-2-hydroxybutanoate + H+ + Nicotinamide adenine dinucleotide phosphate - reduced --> (R)-2,3-Dihydroxy-3-methylpentanoate + Nicotinamide adenine dinucleotide phosphate"
GPR,Cgl1273
Lower bound,0.0
Upper bound,1000.0


## get enzyme parameter

In [5]:
#get kcat_dict and mw_dict
kcat_mw=pd.read_csv(reaction_kcat_MW_file,index_col=0)
kcat_dict={}
mw_dict={}
for i in kcat_mw.index:
    if i in model.reactions:
        kcat_dict[i]=kcat_mw.loc[i,'kcat']
        enz=str(model.reactions.get_by_id(i).gpr)
        flag1=enz.split(' and ')
        flag1.sort(key=None, reverse=False)
        flag2=' and '.join(flag1)
        mw_dict[flag2]=kcat_mw.loc[i,'MW']

In [6]:
kcat_mw

,kcat,MW,kcat_MW
ALAR,9.904612e+05,78.346447,12642.069812
ALATA_L,1.901828e+05,46.748856,4068.179773
ASNN,9.729920e+05,33.298224,29220.536266
ASNS1,7.336000e+03,72.387691,101.343197
ASPT,4.748529e+05,57.567954,8248.562351
...,...,...,...
PSERtex_reverse_num4,2.663573e+07,28.237185,943285.533462
URAt2rpp_reverse_num2,2.663573e+07,66.934412,397937.735374
PHETA1_reverse_num2,1.026000e+04,79.835554,128.514171
SHK3Dr_reverse_num2,4.380570e+05,29.304723,14948.341262


## Add kcat/mw to the dictionary model


In [7]:
def add_parameter(dictionarymodel,kcatdata,kmdata,mwdata):
    for enz in dictionarymodel['enzyme']:
        if enz in mwdata.keys():
            dictionarymodel['enzyme'][enz]['MW']=mwdata[enz]
        for rea in dictionarymodel['enzyme'][enz]['reactions']:
            if  rea in kcatdata.keys():
                dictionarymodel['enzyme'][enz]['reactions'][rea]['kcat']=kcatdata[rea]
            if  rea in kmdata.keys():
                dictionarymodel['enzyme'][enz]['reactions'][rea]['km']=kmdata[rea]
    for enz in mwdata:
        if enz not in dictionarymodel['enzyme']:
            print(enz)

## write the dictionary model to a file

In [8]:
add_parameter(dictionary_model,kcat_dict,{},mw_dict)
json_write(dictionarymodel_path,dictionary_model)